In [0]:
%pip install house_price-0.0.1-py3-none-any.whl

In [0]:
fe = FeatureEngineeringClient()
feature_table_name = f"{catalog_name}.{schema_name}.house_features"

try:
    online_store = fe.get_online_store(name="house-features")
except:
    fe.create_online_store(
        name="house-features",
        capacity="CU_1"
    )
    online_store = fe.get_online_store(name="house-features")

# Publish the feature table to the online store
fe.publish_table(
    online_store=online_store,
    source_table_name=feature_table_name,
    online_table_name=f"{feature_table_name}_online"
)

In [0]:
fe = FeatureEngineeringClient()
# fe.create_online_store(
#     name="house-features",
#     capacity="CU_1"
# )

online_store = fe.get_online_store(name="house-features")
feature_table_name = f"{catalog_name}.{schema_name}.house_features"

fe.publish_table(
    online_store=online_store,
    source_table_name=feature_table_name,
    online_table_name=f"{feature_table_name}_online"
)

In [0]:
fe = FeatureEngineeringClient()
fe.delete_online_store(name="house-features")

In [0]:
%restart_python

In [0]:
import os
import time
from typing import Dict, List

import requests
from loguru import logger
from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession

from house_price.config import ProjectConfig
from house_price.serving.fe_model_serving import FeatureLookupServing

# spark session

spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# get environment variables
os.environ["DBR_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
os.environ["DBR_HOST"] = spark.conf.get("spark.databricks.workspaceUrl")

# Load project config
config = ProjectConfig.from_yaml(config_path="../project_config.yml")
catalog_name = config.catalog_name
schema_name = config.schema_name
endpoint_name = "house-prices-model-serving-fe"

In [0]:
# Initialize Feature Lookup Serving Manager
feature_model_server = FeatureLookupServing(
    model_name=f"{catalog_name}.{schema_name}.house_prices_model_fe",
    endpoint_name=endpoint_name,
    feature_table_name=f"{catalog_name}.{schema_name}.house_features",
)

# Create the online table for house features
# feature_model_server.create_or_update_online_table()

In [0]:
# Deploy the model serving endpoint with feature lookup
feature_model_server.deploy_or_update_serving_endpoint()


In [0]:
# Create a sample request body
required_columns = [
    "LotFrontage",
    "LotArea",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "MasVnrArea",
    "TotalBsmtSF",
    "MSZoning",
    "Street",
    "Alley",
    "LotShape",
    "LandContour",
    "Neighborhood",
    "Condition1",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "CentralAir",
    "SaleType",
    "SaleCondition",
    "Id",
]

spark = SparkSession.builder.getOrCreate()

train_set = spark.table(f"{config.catalog_name}.{config.schema_name}.train_set").toPandas()
sampled_records = train_set[required_columns].sample(n=1000, replace=True).to_dict(orient="records")
dataframe_records = [[record] for record in sampled_records]

logger.info(train_set.dtypes)
logger.info(dataframe_records[0])


In [0]:
house_features = spark.table("mlops_dev.house_prices.house_features").toPandas()

In [0]:
len(house_features)

In [0]:
# Call the endpoint with one sample record
def call_endpoint(record):
    """
    Calls the model serving endpoint with a given input record.
    """
    serving_endpoint = f"https://{os.environ['DBR_HOST']}/serving-endpoints/{endpoint_name}/invocations"

    response = requests.post(
        serving_endpoint,
        headers={"Authorization": f"Bearer {os.environ['DBR_TOKEN']}"},
        json={"dataframe_records": record},
    )
    return response.status_code, response.text

status_code, response_text = call_endpoint(dataframe_records[0])
print(f"Response Status: {status_code}")
print(f"Response Text: {response_text}")

In [0]:
# Load test
for i in range(len(dataframe_records)):
    status_code, response_text = call_endpoint(dataframe_records[i])
    print(f"Response Status: {status_code}")
    print(f"Response Text: {response_text}")
    time.sleep(0.2)